 # 03c_RLVR_Transaction_Math_Verification_v2.py



 ### 🔬 Forensic Mathematical Deconstruction & Bit-Level Blotter Audit

 **Domain Standard: Staggered Multi-Sleeve MTM Architecture**



 Validates the arithmetic integrity of the quantitative trading pipeline:

 1. **Intra-Sleeve Weight Partitioning**: $w_{i} = 1/K_t$ across surviving assets; $\sum w_i \equiv 1.0$.

 2. **Multi-Sleeve MTM Gross Return**: Reconstructs overlapping sleeve queue returns ($r_{\text{stock}, t}$) with zero lookahead bias.

 3. **Dynamic Asset Allocation**: $r_{\text{gross}, t} = w_{\text{active}} r_{\text{stock}} + w_{\text{bm}} r_{\text{bm}} + w_{\text{cash}} r_{\text{cash}}$.

 4. **Amortized Execution Drag & Chunk Restarts**: Slippage deduction and fold queue clearing derived from `run_metadata.json`.

 5. **Ex-Post Linear Ensemble**: Proves $r_{\text{blend}, t} = \frac{1}{M} \sum_{m} r_{\text{net}, t}^{(m)}$.

 6. **Strict Geometric Compounding**: Verifies $V_p(t) = \prod(1+r_{\tau})$ and exact Alpha Multiplier $\alpha(t) = V_p(t) / V_{\text{bm}}(t)$.

 7. **Automated Assertion Suite**: Rigorously verifies bit-level float parity across all out-of-sample sessions.

In [ ]:
# CELL 1: Setup, Dynamic Environment Resolution & Metadata Binding
import json
import os
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import torch

from core.paths import LOCAL_DATA_DIR, OUTPUT_DIR
from core.settings import CacheConfig, TradingConfig

# Environment / Colab Auto-Resolution
RUNNING_IN_COLAB = "google.colab" in sys.modules
if RUNNING_IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    rl_root = Path(
        "/content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v3"
    )
else:
    rl_root_default = Path(
        r"C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3"
    )
    rl_root = rl_root_default if rl_root_default.exists() else Path.cwd()

if str(rl_root) not in sys.path:
    sys.path.append(str(rl_root))
os.chdir(rl_root)

# Dynamic Resolution of run_metadata.json
TARGET_GEN: Optional[int] = 25  # Set to target generation or None for latest discovery
checkpoint_base = OUTPUT_DIR / "model_checkpoints"

metadata_path: Optional[Path] = None
candidate_paths = [
    (
        checkpoint_base / f"walk_forward_gen{TARGET_GEN}" / "run_metadata.json"
        if TARGET_GEN
        else None
    ),
    OUTPUT_DIR / f"run_metadata_gen{TARGET_GEN}.json" if TARGET_GEN else None,
    (
        OUTPUT_DIR / "canonical_anchors" / f"run_metadata_gen{TARGET_GEN}.json"
        if TARGET_GEN
        else None
    ),
    OUTPUT_DIR / "run_metadata.json",
    checkpoint_base / "run_metadata.json",
]

for cp in candidate_paths:
    if cp is not None and cp.exists():
        metadata_path = cp
        break

if metadata_path is None:
    # Broad recursive search
    matches = sorted(OUTPUT_DIR.rglob("*run_metadata*.json"))
    if matches:
        metadata_path = matches[-1]
    else:
        raise FileNotFoundError(
            "❌ run_metadata.json could not be resolved in OUTPUT_DIR."
        )

with open(metadata_path, "r", encoding="utf-8") as f:
    run_metadata: Dict[str, Any] = json.load(f)

# Extract Dynamic Contracts from Metadata
GEN: int = int(run_metadata.get("generation", TARGET_GEN if TARGET_GEN else 25))
SEEDS: List[int] = [int(s) for s in run_metadata.get("seeds", [42, 101, 777])]
CHUNK_SPECS: List[Dict[str, Any]] = run_metadata.get("chunk_specs", [])

# Extract Chronological Chunk Deployment Dates Dynamically
CHUNK_DEPLOY_STARTS: List[pd.Timestamp] = [
    pd.Timestamp(c["deploy_start"])
    for c in CHUNK_SPECS
    if c.get("deploy_start") is not None
]

# Instantiate Institutional Trading Configuration and Synchronize with Metadata
config = TradingConfig()
meta_cfg = run_metadata.get("trading_config", {})
if meta_cfg:
    for field in [
        "benchmark",
        "benchmark_ticker",
        "holding_period",
        "slippage_rate",
        "max_cash_pct",
        "min_active_tilt",
        "min_basket_width",
    ]:
        if field in meta_cfg and hasattr(config, field):
            setattr(config, field, meta_cfg[field])

benchmark_ticker: str = config.benchmark
H: int = int(config.holding_period)
slippage_rate: float = float(config.slippage_rate)
max_cash_pct: float = float(config.max_cash_pct)
min_active_tilt: float = float(config.min_active_tilt)
min_basket_width: int = int(config.min_basket_width)

print("=" * 85)
print("🏛️ INSTITUTIONAL TRADING ENGINE CONTRACTS LOADED (DYNAMIC METADATA BINDING)")
print(f"  • Source Metadata Manifest    : {metadata_path}")
print(f"  • Generation Tag (GEN)        : {GEN}")
print(f"  • Random Ensemble Seeds       : {SEEDS}")
print(f"  • Chunk Folds Ingested        : {len(CHUNK_DEPLOY_STARTS)} deployments")
for idx, dt in enumerate(CHUNK_DEPLOY_STARTS):
    print(f"      - Chunk {idx} Deploy Start : {dt.strftime('%Y-%m-%d')}")
print(f"  • Benchmark Ticker (Dynamic)  : {benchmark_ticker}")
print(f"  • Staggered Sleeve Horizon H  : {H} sessions")
print(f"  • Minimum Basket Width Floor  : {min_basket_width} tickers")
print(f"  • Amortized Slippage Rate     : {slippage_rate * 10000:.1f} bps")
print(
    f"  • Max Cash Permitted          : {max_cash_pct * 100:.1f}% (Zero Cash Mandate)"
)
print(f"  • Minimum Active Tilt Floor   : {min_active_tilt * 100:.1f}%")
print("=" * 85)

NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3
PROJECT_ROOT: C:\Users\ping\Files_win10\python\py311\stocks
GLOBAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data
GLOBAL_PROCESSED_DIR: C:\Users\ping\Files_win10\python\py311\stocks\data\processed
LOCAL_DATA_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\data
OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\output
TRADES_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\trades

🏛️ INSTITUTIONAL TRADING ENGINE CONTRACTS LOADED (DYNAMIC METADATA BINDING)
  • Source Metadata Manifest    : C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v3\output\model_checkpoints\walk_forward_gen25\run_metadata.json
  • Generation Tag (GEN)        : 25
  • Random Ensemble Seeds       : [42, 101, 777]
  • Chunk Folds Ingested        : 5 deployments
      - Chunk 0 Deploy Start : 2022-03-25
      - Chunk 1 Deploy Start : 2023-03-2

In [ ]:
# CELL 2: Ingest Continuous Blotters (Blend & Constituent Seeds) with Dynamic Lineage
constituent_filenames: Dict[int, str] = {}
constituent_dfs: Dict[int, pd.DataFrame] = {}


def resolve_artifact_path(
    expected_filename: str, meta_hint: Optional[str] = None
) -> Path:
    """Robust multi-environment resolver for parquet blotters."""
    if meta_hint:
        p = Path(meta_hint)
        if p.exists():
            return p
        # Check relative to local rl_root
        local_cand = rl_root / "output" / p.name
        if local_cand.exists():
            return local_cand
        canon_cand = rl_root / "output" / "canonical_anchors" / p.name
        if canon_cand.exists():
            return canon_cand

    search_paths = [
        OUTPUT_DIR / expected_filename,
        OUTPUT_DIR / "canonical_anchors" / expected_filename,
        checkpoint_base / expected_filename,
        checkpoint_base / f"walk_forward_gen{GEN}" / expected_filename,
    ]
    for sp in search_paths:
        if sp.exists():
            return sp

    matches = list(OUTPUT_DIR.rglob(expected_filename))
    if matches:
        return matches[0]
    raise FileNotFoundError(
        f"❌ Required blotter artifact not found: {expected_filename}"
    )


artifacts_meta = run_metadata.get("artifacts", {})
constituent_meta_map = artifacts_meta.get("constituent_blotters", {})

for s in SEEDS:
    filename = f"blotter_continuous_s{s}_gen{GEN}.parquet"
    constituent_filenames[s] = filename
    hint = constituent_meta_map.get(str(s), constituent_meta_map.get(s, None))
    resolved_path = resolve_artifact_path(filename, hint)

    df = pd.read_parquet(resolved_path)
    date_col = "date" if "date" in df.columns else "decision_date"
    df["date"] = pd.to_datetime(df[date_col])
    df = df.sort_values("date").reset_index(drop=True)
    constituent_dfs[s] = df
    print(
        f"✅ Loaded Seed {s:03d} [{resolved_path.name}]: {len(df)} sessions "
        f"({df['date'].min().strftime('%Y-%m-%d')} -> {df['date'].max().strftime('%Y-%m-%d')})"
    )

blend_filename = f"blotter_continuous_ex_post_blend_gen{GEN}.parquet"
blend_hint = artifacts_meta.get(
    "blend_blotter", artifacts_meta.get("canonical_blend_blotter", None)
)
blend_path = resolve_artifact_path(blend_filename, blend_hint)

blend_df = pd.read_parquet(blend_path)
b_date_col = "date" if "date" in blend_df.columns else "decision_date"
blend_df["date"] = pd.to_datetime(blend_df[b_date_col])
blend_df = blend_df.sort_values("date").reset_index(drop=True)
print(f"✅ Loaded Ex-Post Blend [{blend_path.name}]: {len(blend_df)} sessions")

# Rigorous Temporal Alignment Verification
N_SESSIONS = len(blend_df)
for s, df in constituent_dfs.items():
    assert (
        len(df) == N_SESSIONS
    ), f"Length mismatch for seed {s}: {len(df)} vs {N_SESSIONS}"
    assert (
        df["date"].values == blend_df["date"].values
    ).all(), f"Date alignment mismatch for seed {s}"

print(
    f"🎯 Verified 100% calendar parity across all {N_SESSIONS} walk-forward sessions."
)

✅ Loaded Seed 042 [blotter_continuous_s42_gen25.parquet]: 1120 sessions (2022-03-25 -> 2026-09-23)
✅ Loaded Seed 101 [blotter_continuous_s101_gen25.parquet]: 1120 sessions (2022-03-25 -> 2026-09-23)
✅ Loaded Seed 777 [blotter_continuous_s777_gen25.parquet]: 1120 sessions (2022-03-25 -> 2026-09-23)
✅ Loaded Ex-Post Blend [blotter_continuous_ex_post_blend_gen25.parquet]: 1120 sessions
🎯 Verified 100% calendar parity across all 1120 walk-forward sessions.


In [ ]:
# CELL 3: First-Principles Deconstruction & Assertion Across All Constituent Seeds
seed_math_audits: Dict[int, Dict[str, float]] = {}

for s in SEEDS:
    df = constituent_dfs[s]

    # 1. Intra-Sleeve Uniform Partitioning: w_i = 1 / K_t
    if "tickers" in df.columns:
        for idx, row in df.iterrows():
            tickers = row["tickers"]
            if isinstance(tickers, (list, np.ndarray)) and len(tickers) > 0:
                k_actual = len(tickers)
                assert k_actual >= 1, f"Row {idx}: basket is unexpectedly empty"
                w_uniform = 1.0 / k_actual
                sum_w = w_uniform * k_actual
                assert (
                    abs(sum_w - 1.0) < 1e-12
                ), f"Row {idx}: intra-sleeve weights != 1.0"

    # Extract blotter arrays
    w_active = df["weight_active"].to_numpy(dtype=float)
    w_bm = df["weight_benchmark"].to_numpy(dtype=float)
    w_cash = df["weight_cash"].to_numpy(dtype=float)
    r_stock = df["gross_stock_daily_simple_ret"].to_numpy(dtype=float)
    r_bm = df["bm_daily_simple_ret"].to_numpy(dtype=float)
    r_cash = df.get("cash_daily_simple_ret", pd.Series(0.0, index=df.index)).to_numpy(
        dtype=float
    )
    stored_gross = df["gross_daily_simple_ret"].to_numpy(dtype=float)
    stored_slippage = df["slippage_daily_simple_loss"].to_numpy(dtype=float)
    stored_net = df["net_daily_simple_ret"].to_numpy(dtype=float)
    stored_alpha = df["alpha_daily_simple_ret"].to_numpy(dtype=float)
    stored_vp = df["agent_equity"].to_numpy(dtype=float)
    stored_vbm = df["bm_equity"].to_numpy(dtype=float)
    stored_alpha_eq = df["alpha_equity"].to_numpy(dtype=float)

    # Invariants Verification
    assert np.all(
        w_cash == 0.0
    ), f"Seed {s}: Cash hoarding detected! Expected w_cash == 0.0"
    np.testing.assert_allclose(
        w_active + w_bm + w_cash,
        1.0,
        atol=1e-7,
        err_msg=f"Seed {s}: Capital allocation != 1.0",
    )

    # 2. Gross Return Recomputation
    r_gross_calc = np.empty_like(r_bm)
    r_gross_calc[0] = r_bm[0]
    r_gross_calc[1:] = (
        w_active[1:] * r_stock[1:] + w_bm[1:] * r_bm[1:] + w_cash[1:] * r_cash[1:]
    )

    # 3. Slippage Recomputation (Aware of Walk-Forward Chunk Boundaries)
    is_active_rotation = stored_slippage > 0.0
    slippage_calc = np.where(is_active_rotation, (w_active / H) * slippage_rate, 0.0)
    assert np.all(stored_slippage[~is_active_rotation] == 0.0)

    # 4. Net Strategy Return
    r_net_calc = r_gross_calc - slippage_calc

    # 5. Net Daily Alpha Spread
    r_alpha_calc = r_net_calc - r_bm

    # 6. Cumulative Discrete Compounding Wealth Curves
    v_p_calc = np.cumprod(1.0 + r_net_calc)
    v_bm_calc = np.cumprod(1.0 + r_bm)
    alpha_eq_calc = v_p_calc / v_bm_calc

    # 7. Bit-Level Forensic Deltas
    delta_gross = float(np.max(np.abs(r_gross_calc - stored_gross)))
    delta_slippage = float(np.max(np.abs(slippage_calc - stored_slippage)))
    delta_net = float(np.max(np.abs(r_net_calc - stored_net)))
    delta_alpha = float(np.max(np.abs(r_alpha_calc - stored_alpha)))
    delta_vp = float(np.max(np.abs(v_p_calc - stored_vp)))
    delta_vbm = float(np.max(np.abs(v_bm_calc - stored_vbm)))
    delta_alpha_eq = float(np.max(np.abs(alpha_eq_calc - stored_alpha_eq)))

    # Machine Precision Assertions (atol = 1e-7)
    assert delta_gross < 1e-7, f"Seed {s}: Gross return delta {delta_gross:.2e} >= 1e-7"
    assert (
        delta_slippage < 1e-7
    ), f"Seed {s}: Slippage delta {delta_slippage:.2e} >= 1e-7"
    assert delta_net < 1e-7, f"Seed {s}: Net return delta {delta_net:.2e} >= 1e-7"
    assert delta_alpha < 1e-7, f"Seed {s}: Daily alpha delta {delta_alpha:.2e} >= 1e-7"
    assert delta_vp < 1e-7, f"Seed {s}: Equity curve delta {delta_vp:.2e} >= 1e-7"
    assert delta_vbm < 1e-7, f"Seed {s}: Benchmark curve delta {delta_vbm:.2e} >= 1e-7"
    assert (
        delta_alpha_eq < 1e-7
    ), f"Seed {s}: Alpha equity delta {delta_alpha_eq:.2e} >= 1e-7"

    seed_math_audits[s] = {
        "Max |Δ Gross Return|": delta_gross,
        "Max |Δ Slippage|": delta_slippage,
        "Max |Δ Net Return|": delta_net,
        "Max |Δ Daily Alpha|": delta_alpha,
        "Max |Δ Agent Equity|": delta_vp,
        "Max |Δ BM Equity|": delta_vbm,
        "Max |Δ Alpha Multiplier|": delta_alpha_eq,
    }

print("=" * 85)
print("🛡️ CONSTITUENT SEED FORENSIC AUDIT: BIT-LEVEL PARITY")
print("=" * 85)
audit_summary = pd.DataFrame(seed_math_audits).T
for col in audit_summary.columns:
    print(
        f"  • {col:<26}: Max across seeds = {audit_summary[col].max():.2e} (PASSED < 1e-7)"
    )
print("=" * 85)

🛡️ CONSTITUENT SEED FORENSIC AUDIT: BIT-LEVEL PARITY
  • Max |Δ Gross Return|      : Max across seeds = 0.00e+00 (PASSED < 1e-7)
  • Max |Δ Slippage|          : Max across seeds = 2.71e-20 (PASSED < 1e-7)
  • Max |Δ Net Return|        : Max across seeds = 3.47e-18 (PASSED < 1e-7)
  • Max |Δ Daily Alpha|       : Max across seeds = 3.47e-18 (PASSED < 1e-7)
  • Max |Δ Agent Equity|      : Max across seeds = 0.00e+00 (PASSED < 1e-7)
  • Max |Δ BM Equity|         : Max across seeds = 0.00e+00 (PASSED < 1e-7)
  • Max |Δ Alpha Multiplier|  : Max across seeds = 0.00e+00 (PASSED < 1e-7)


In [ ]:
# CELL 4: Ex-Post Linear Blend Ensemble Recomputation & Invariant Proof
seed_net_matrix = np.column_stack(
    [constituent_dfs[s]["net_daily_simple_ret"].to_numpy(dtype=float) for s in SEEDS]
)
blend_r_net_recalc = np.mean(seed_net_matrix, axis=1)

# Benchmark Parity: All blotters must share identical benchmark return stream
stored_bm_ret = blend_df["bm_daily_simple_ret"].to_numpy(dtype=float)
for s in SEEDS:
    seed_bm = constituent_dfs[s]["bm_daily_simple_ret"].to_numpy(dtype=float)
    assert np.all(seed_bm == stored_bm_ret), f"Benchmark stream discrepancy in seed {s}"

# Blend Alpha Spread: r_blend_alpha,t = r_blend,t - r_bm,t
blend_r_alpha_recalc = blend_r_net_recalc - stored_bm_ret

# Exact Geometric Compounding
blend_vp_recalc = np.cumprod(1.0 + blend_r_net_recalc)
blend_vbm_recalc = np.cumprod(1.0 + stored_bm_ret)
blend_alpha_eq_recalc = blend_vp_recalc / blend_vbm_recalc

# Assertions against stored blend parquet
stored_blend_net = blend_df["net_daily_simple_ret"].to_numpy(dtype=float)
stored_blend_alpha = blend_df["alpha_daily_simple_ret"].to_numpy(dtype=float)
stored_blend_vp = blend_df["agent_equity"].to_numpy(dtype=float)
stored_blend_vbm = blend_df["bm_equity"].to_numpy(dtype=float)
stored_blend_alpha_eq = blend_df["alpha_equity"].to_numpy(dtype=float)

max_d_blend_net = float(np.max(np.abs(blend_r_net_recalc - stored_blend_net)))
max_d_blend_alpha = float(np.max(np.abs(blend_r_alpha_recalc - stored_blend_alpha)))
max_d_blend_vp = float(np.max(np.abs(blend_vp_recalc - stored_blend_vp)))
max_d_blend_vbm = float(np.max(np.abs(blend_vbm_recalc - stored_blend_vbm)))
max_d_blend_alpha_eq = float(
    np.max(np.abs(blend_alpha_eq_recalc - stored_blend_alpha_eq))
)

assert max_d_blend_net < 1e-7, f"Blend net return delta {max_d_blend_net:.2e} >= 1e-7"
assert (
    max_d_blend_alpha < 1e-7
), f"Blend daily alpha delta {max_d_blend_alpha:.2e} >= 1e-7"
assert max_d_blend_vp < 1e-7, f"Blend equity delta {max_d_blend_vp:.2e} >= 1e-7"
assert max_d_blend_vbm < 1e-7, f"Blend BM equity delta {max_d_blend_vbm:.2e} >= 1e-7"
assert (
    max_d_blend_alpha_eq < 1e-7
), f"Blend alpha equity delta {max_d_blend_alpha_eq:.2e} >= 1e-7"

print("=" * 85)
print(f"🏆 EX-POST LINEAR BLEND FORENSIC VERIFICATION ({blend_filename})")
print("=" * 85)
print(
    f"  • Max |Δ Blend Net Return|      : {max_d_blend_net:.2e} (Strict Bit-Level Parity)"
)
print(
    f"  • Max |Δ Blend Daily Alpha|     : {max_d_blend_alpha:.2e} (Strict Bit-Level Parity)"
)
print(
    f"  • Max |Δ Blend Agent Wealth Vp| : {max_d_blend_vp:.2e} (Zero Compounding Drift)"
)
print(
    f"  • Max |Δ Blend Benchmark Vbm|   : {max_d_blend_vbm:.2e} (Zero Compounding Drift)"
)
print(
    f"  • Max |Δ Alpha Multiplier|      : {max_d_blend_alpha_eq:.2e} (Strict Geometric Ratio)"
)
print("=" * 85)

🏆 EX-POST LINEAR BLEND FORENSIC VERIFICATION (blotter_continuous_ex_post_blend_gen25.parquet)
  • Max |Δ Blend Net Return|      : 0.00e+00 (Strict Bit-Level Parity)
  • Max |Δ Blend Daily Alpha|     : 0.00e+00 (Strict Bit-Level Parity)
  • Max |Δ Blend Agent Wealth Vp| : 0.00e+00 (Zero Compounding Drift)
  • Max |Δ Blend Benchmark Vbm|   : 0.00e+00 (Zero Compounding Drift)
  • Max |Δ Alpha Multiplier|      : 0.00e+00 (Strict Geometric Ratio)


In [ ]:
# CELL 5: Bit-Level Multi-Sleeve MTM Queue Reconstruction from Raw Prices
# Fully dynamic fold restart detection via metadata chunk specifications
from core.accounting import MTMPortfolioEngine
from data_pipeline.loader import load_processed_data

print("=" * 85)
print("⚡ DIRECT MULTI-SLEEVE MTM QUEUE RECONSTRUCTION FROM RAW OHLCV PRICES")
print("=" * 85)

raw_data = load_processed_data()
df_close = raw_data.df_ohlcv["Adj Close"].unstack(level=0).sort_index()
df_close.index = pd.to_datetime(df_close.index)

# Next-day execution return: Close(T+1) to Close(T+2) relative to Decision Day T
forward_rets = df_close.astype(float).pct_change(1, fill_method=None).shift(-1)

for s in SEEDS:
    s_df = constituent_dfs[s].copy()
    s_df["date"] = pd.to_datetime(s_df["date"])

    # Dynamically resolve restart session indices for all chunk deployments
    chunk_restart_indices = set()
    for start_dt in CHUNK_DEPLOY_STARTS:
        matching_indices = s_df.index[s_df["date"] >= start_dt]
        if len(matching_indices) > 0:
            chunk_restart_indices.add(matching_indices[0])

    reconstructed_gross_stock = np.zeros(len(s_df), dtype=float)
    engine = MTMPortfolioEngine(config)

    for t_idx in range(len(s_df)):
        # Reset queue at every walk-forward deployment boundary (Chunk 1, 2, 3, 4, ...)
        if t_idx in chunk_restart_indices and t_idx > 0:
            engine.reset()

        cur_date = s_df["date"].iloc[t_idx]
        row_tickers = s_df["tickers"].iloc[t_idx]
        selected = (
            list(row_tickers)
            if isinstance(row_tickers, (list, np.ndarray))
            else ([] if pd.isna(row_tickers) else [row_tickers])
        )

        day_rets = forward_rets.loc[cur_date] if cur_date in forward_rets.index else {}
        bm_ret = float(s_df["bm_daily_simple_ret"].iloc[t_idx])
        eq_exp = float(s_df["equity_exposure"].iloc[t_idx])
        act_tilt = float(s_df["active_tilt"].iloc[t_idx])

        step_res = engine.step(
            selected_tickers=selected,
            equity_exposure=eq_exp,
            active_tilt=act_tilt,
            stock_simple_rets=day_rets,
            bm_daily_simple_ret=bm_ret,
        )
        reconstructed_gross_stock[t_idx] = step_res.gross_stock_daily_simple_ret

    stored_gross_stock = s_df["gross_stock_daily_simple_ret"].to_numpy(dtype=float)
    abs_deltas = np.abs(reconstructed_gross_stock - stored_gross_stock)
    delta_stock = float(np.max(abs_deltas))

    print(
        f"  • Seed {s:03d} Max |Δ Reconstructed Gross Stock - Blotter Gross Stock|: {delta_stock:.2e}"
    )
    assert delta_stock < 1e-5, (
        f"Seed {s}: Price reconstruction parity gate failed! "
        f"Delta: {delta_stock:.2e} (threshold: 1e-5)"
    )

print("=" * 85)
print(
    "🏆 VERIFIED: Bit-Level Parity confirmed between Raw OHLCV Prices and Blotters across ALL seeds!"
)
print("=" * 85)

⚡ DIRECT MULTI-SLEEVE MTM QUEUE RECONSTRUCTION FROM RAW OHLCV PRICES
  • Seed 042 Max |Δ Reconstructed Gross Stock - Blotter Gross Stock|: 9.85e-10
  • Seed 101 Max |Δ Reconstructed Gross Stock - Blotter Gross Stock|: 0.00e+00
  • Seed 777 Max |Δ Reconstructed Gross Stock - Blotter Gross Stock|: 5.45e-10
🏆 VERIFIED: Bit-Level Parity confirmed between Raw OHLCV Prices and Blotters across ALL seeds!


In [ ]:
# CELL 6: Institutional Parity Inspection Table
def generate_audit_row(df: pd.DataFrame, sample_idx: int, label: str) -> pd.DataFrame:
    n_rows = len(df)
    actual_idx = sample_idx % n_rows
    r = df.iloc[actual_idx]
    date_str = pd.to_datetime(r["date"]).strftime("%Y-%m-%d")

    w_act = float(r.get("weight_active", 1.0))
    w_bm = float(r.get("weight_benchmark", 0.0))
    w_csh = float(r.get("weight_cash", 0.0))
    r_stk = float(r.get("gross_stock_daily_simple_ret", r.get("net_daily_simple_ret")))
    r_bm = float(r.get("bm_daily_simple_ret", 0.0))
    slip = float(r.get("slippage_daily_simple_loss", 0.0))

    is_launch = actual_idx == 0
    hand_gross = r_bm if is_launch else (w_act * r_stk + w_bm * r_bm + w_csh * 0.0)
    hand_slip = (w_act / H) * slippage_rate if slip > 0.0 else 0.0
    hand_net = hand_gross - hand_slip
    hand_alpha = hand_net - r_bm

    stored_gross = float(r.get("gross_daily_simple_ret", r.get("net_daily_simple_ret")))
    stored_slip = slip
    stored_net = float(r["net_daily_simple_ret"])
    stored_alpha = float(r["alpha_daily_simple_ret"])
    stored_vp = float(r["agent_equity"])
    stored_vbm = float(r["bm_equity"])
    stored_alpha_eq = float(r["alpha_equity"])

    sub_net = df["net_daily_simple_ret"].iloc[: actual_idx + 1].to_numpy(dtype=float)
    sub_bm = df["bm_daily_simple_ret"].iloc[: actual_idx + 1].to_numpy(dtype=float)
    hand_vp = float(np.prod(1.0 + sub_net))
    hand_vbm = float(np.prod(1.0 + sub_bm))
    hand_alpha_eq = hand_vp / hand_vbm

    metrics = [
        ("Active Stock Allocation (w_active)", w_act, w_act),
        ("Benchmark Shelter Allocation (w_bm)", w_bm, w_bm),
        ("Cash Allocation (w_cash)", w_csh, 0.0),
        ("Gross Portfolio Return (r_gross)", stored_gross, hand_gross),
        ("Amortized Slippage Loss", stored_slip, hand_slip),
        ("Net Portfolio Return (r_net)", stored_net, hand_net),
        ("Net Daily Alpha Spread", stored_alpha, hand_alpha),
        ("Portfolio Wealth Multiplier (V_p)", stored_vp, hand_vp),
        ("Benchmark Wealth Multiplier (V_bm)", stored_vbm, hand_vbm),
        ("Strict Alpha Multiplier (V_p / V_bm)", stored_alpha_eq, hand_alpha_eq),
    ]

    records = []
    for name, st, hand in metrics:
        delta = abs(st - hand)
        records.append(
            {
                "Inspection Session": f"{label} ({date_str})",
                "Metric": name,
                "Blotter Value": f"{st:.6f}",
                "First-Principles Calc": f"{hand:.6f}",
                "Absolute Delta": f"{delta:.2e}",
                "Bit-Level Parity (<1e-7)": "PASS ✅" if delta < 1e-7 else "FAIL ❌",
            }
        )
    return pd.DataFrame(records)


audit_seed = SEEDS[0]
print("\n" + "=" * 95)
print(
    f"📋 INSTITUTIONAL PARITY AUDIT: 3 CRITICAL LIFECYCLE SESSIONS (SEED {audit_seed} - GEN {GEN})"
)
print("=" * 95)

table_launch = generate_audit_row(constituent_dfs[audit_seed], 0, "1. Launch Session")
table_mid = generate_audit_row(
    constituent_dfs[audit_seed],
    len(constituent_dfs[audit_seed]) // 2,
    "2. Midpoint Rebalance",
)
table_term = generate_audit_row(
    constituent_dfs[audit_seed], -1, "3. Terminal Out-of-Sample"
)

full_audit_table = pd.concat([table_launch, table_mid, table_term], ignore_index=True)
print(full_audit_table.to_string(index=False))
print("=" * 95)


📋 INSTITUTIONAL PARITY AUDIT: 3 CRITICAL LIFECYCLE SESSIONS (SEED 42 - GEN 25)
                    Inspection Session                               Metric Blotter Value First-Principles Calc Absolute Delta Bit-Level Parity (<1e-7)
        1. Launch Session (2022-03-25)   Active Stock Allocation (w_active)      0.867867              0.867867       0.00e+00                   PASS ✅
        1. Launch Session (2022-03-25)  Benchmark Shelter Allocation (w_bm)      0.132133              0.132133       0.00e+00                   PASS ✅
        1. Launch Session (2022-03-25)             Cash Allocation (w_cash)      0.000000              0.000000       0.00e+00                   PASS ✅
        1. Launch Session (2022-03-25)     Gross Portfolio Return (r_gross)      0.007112              0.007112       0.00e+00                   PASS ✅
        1. Launch Session (2022-03-25)              Amortized Slippage Loss      0.000000              0.000000       0.00e+00                   PASS ✅
        

In [ ]:
# CELL 7: Visual Forensic Proof: Centered Header & Right-Rail Legend Layout
total_blend_ret = (blend_df["agent_equity"].iloc[-1] - 1.0) * 100
total_bm_ret = (blend_df["bm_equity"].iloc[-1] - 1.0) * 100

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.07,
    row_heights=[0.45, 0.35, 0.20],
    subplot_titles=(
        f"📈 Realized Trajectories: Ex-Post Blend vs {benchmark_ticker}",
        f"🔥 Cumulative Geometric Alpha Multiplier (V_p / V_{benchmark_ticker})",
        "🔬 Bit-Level Accumulation Parity Residue log10(|V_calc - V_blotter|)",
    ),
)

# Row 1: Cumulative Wealth Trajectories
fig.add_trace(
    go.Scatter(
        x=blend_df["date"],
        y=blend_df["agent_equity"],
        mode="lines",
        name=f"Ex-Post Blend [{total_blend_ret:+.1f}%]",
        line=dict(color="#00CC96", width=2.5),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=blend_df["date"],
        y=blend_df["bm_equity"],
        mode="lines",
        name=f"Benchmark ({benchmark_ticker}) [{total_bm_ret:+.1f}%]",
        line=dict(color="#111827", width=2.0, dash="dash"),
    ),
    row=1,
    col=1,
)
for s in SEEDS:
    s_ret = (constituent_dfs[s]["agent_equity"].iloc[-1] - 1.0) * 100
    fig.add_trace(
        go.Scatter(
            x=constituent_dfs[s]["date"],
            y=constituent_dfs[s]["agent_equity"],
            mode="lines",
            name=f"Seed {s:03d} [{s_ret:+.1f}%]",
            opacity=0.45,
            line=dict(width=1.2),
        ),
        row=1,
        col=1,
    )

# Row 2: Alpha Multipliers (Agent V_p / BM V_bm)
fig.add_trace(
    go.Scatter(
        x=blend_df["date"],
        y=blend_df["alpha_equity"],
        mode="lines",
        name="Alpha Multiplier",
        line=dict(color="#636EFA", width=2.2),
    ),
    row=2,
    col=1,
)
fig.add_hline(y=1.0, line_dash="dot", line_color="gray", row=2, col=1)

# Row 3: Machine Precision Log Residuals
calc_resid = np.abs(blend_vp_recalc - stored_blend_vp)
log_resid = np.log10(np.maximum(calc_resid, 1e-16))
fig.add_trace(
    go.Scatter(
        x=blend_df["date"],
        y=log_resid,
        mode="lines",
        name="log10(|Δ V_p|)",
        line=dict(color="crimson", width=1.0),
    ),
    row=3,
    col=1,
)
fig.add_hline(
    y=-7.0,
    line_dash="dash",
    line_color="black",
    annotation_text="Strict 1e-7 Gate",
    row=3,
    col=1,
)

seeds_str = ", ".join([f"s{s}" for s in SEEDS])
fig.update_layout(
    height=950,
    width=1240,
    hovermode="x unified",
    margin=dict(t=90, b=40, l=60, r=210),
    title=dict(
        text=(
            "<span style='display: block; text-align: center;'>"
            "<b>Forensic Verification Suite: Exact Geometric Compounding & Residual Tracking</b><br>"
            f"<span style='font-size: 12px; color: #4B5563;'>Blotter: <b>{blend_filename}</b> "
            f"| Constituents: <b>Gen {GEN} ({seeds_str})</b> | Benchmark: <b>{benchmark_ticker}</b></span>"
            "</span>"
        ),
        x=0.44,
        y=0.97,
        xanchor="center",
        yanchor="top",
    ),
    legend=dict(
        orientation="v",
        x=1.01,
        y=1.0,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255, 255, 255, 0.7)",
        bordercolor="#E5E7EB",
        borderwidth=1,
    ),
)
fig.update_yaxes(title_text="Wealth Multiplier", row=1, col=1)
fig.update_yaxes(title_text="Alpha Multiplier", row=2, col=1)
fig.update_yaxes(title_text="log10(Delta)", row=3, col=1)
fig.show()

In [ ]:
# CELL 8: Regime Attribution & True Net Return Correlation
first_deploy_dt = CHUNK_DEPLOY_STARTS[0].strftime("%Y-%m-%d")
mid_deploy_dt = (
    CHUNK_DEPLOY_STARTS[2].strftime("%Y-%m-%d")
    if len(CHUNK_DEPLOY_STARTS) > 2
    else CHUNK_DEPLOY_STARTS[-1].strftime("%Y-%m-%d")
)
mid_end_dt = (pd.Timestamp(mid_deploy_dt) - pd.Timedelta(days=1)).strftime("%Y-%m-%d")

periods = {
    f"Full OOS ({first_deploy_dt[:4]}-Present)": slice(first_deploy_dt, None),
    f"Early Era (Chunks 0-1: {first_deploy_dt[:4]}-{mid_deploy_dt[:4]})": slice(
        first_deploy_dt, mid_end_dt
    ),
    f"Expansion Era (Chunks 2+: {mid_deploy_dt[:4]}-Present)": slice(
        mid_deploy_dt, None
    ),
}

ret_col = "net_daily_simple_ret"
bm_col = "bm_daily_simple_ret"
tilt_col = "weight_active"
ref_seed = SEEDS[0]

for period_name, date_slice in periods.items():
    print(f"\n{'='*22} {period_name} {'='*22}")
    bm_ret = (
        constituent_dfs[ref_seed].set_index("date").loc[date_slice, bm_col].dropna()
    )
    bm_cum = (1.0 + bm_ret).prod() - 1.0
    bm_sharpe = (bm_ret.mean() / (bm_ret.std() + 1e-9)) * np.sqrt(252)
    print(
        f"Benchmark ({benchmark_ticker}) -> Return: {bm_cum*100:6.1f}% | Sharpe: {bm_sharpe:4.2f}"
    )
    print("-" * 65)

    for s in SEEDS:
        sub = constituent_dfs[s].set_index("date").loc[date_slice]
        r = sub[ret_col].dropna()
        cum_ret = (1.0 + r).prod() - 1.0
        sharpe = (r.mean() / (r.std() + 1e-9)) * np.sqrt(252)
        tilt = sub[tilt_col].dropna().mean()
        print(
            f"Seed {s:3d} (Gen {GEN}) -> Net Return: {cum_ret*100:6.1f}% | Sharpe: {sharpe:4.2f} | Tilt: {tilt:.3f}"
        )

# Correlation Matrix over Recent Expansion Era
expansion_slice = slice(mid_deploy_dt, None)
ret_matrix = pd.DataFrame(
    {
        f"Seed_{s}": constituent_dfs[s].set_index("date").loc[expansion_slice, ret_col]
        for s in SEEDS
    }
)
ret_matrix[benchmark_ticker] = (
    constituent_dfs[ref_seed].set_index("date").loc[expansion_slice, bm_col]
)

print(
    f"\n{'='*20} True Net Return Correlation ({mid_deploy_dt[:4]}-Present Expansion) {'='*20}"
)
print(ret_matrix.dropna().corr().round(3))


====================== Full OOS (2022-Present) ======================
Benchmark (SPY) -> Return:   75.4% | Sharpe: 0.82
-----------------------------------------------------------------
Seed  42 (Gen 25) -> Net Return:   66.2% | Sharpe: 0.64 | Tilt: 0.650
Seed 101 (Gen 25) -> Net Return:  159.4% | Sharpe: 0.89 | Tilt: 0.710
Seed 777 (Gen 25) -> Net Return:  147.0% | Sharpe: 0.79 | Tilt: 0.676

====================== Early Era (Chunks 0-1: 2022-2024) ======================
Benchmark (SPY) -> Return:   18.0% | Sharpe: 0.55
-----------------------------------------------------------------
Seed  42 (Gen 25) -> Net Return:   16.0% | Sharpe: 0.43 | Tilt: 0.687
Seed 101 (Gen 25) -> Net Return:   42.9% | Sharpe: 0.72 | Tilt: 0.683
Seed 777 (Gen 25) -> Net Return:   14.0% | Sharpe: 0.37 | Tilt: 0.646

====================== Expansion Era (Chunks 2+: 2024-Present) ======================
Benchmark (SPY) -> Return:   48.6% | Sharpe: 1.08
-----------------------------------------------------------